## Example 1: Basic event notification

In [ ]:
import zope.event

# Example 1: Basic event notification
print("Basic event notification:")

# List to track received events
received_events = []

def event_handler(event):
    """Simple event handler"""
    received_events.append(event)
    print(f"Received event: {event}")

# Subscribe to events
old_subscribers = zope.event.subscribers[:]
zope.event.subscribers.append(event_handler)

print(f"Event handler registered")
print(f"Current subscribers: {len(zope.event.subscribers)}")

# Notify events
zope.event.notify("Hello, World!")
zope.event.notify("Second event")
zope.event.notify({"type": "data", "value": 42})

print(f"Total events received: {len(received_events)}")
print(f"Events: {received_events}")

# Cleanup
zope.event.subscribers[:] = old_subscribers

print("Basic event notification completed")

## Example 2: Multiple event handlers

In [ ]:
import zope.event

# Example 2: Multiple event handlers
print("Multiple event handlers:")

# Different handlers for different purposes
log_entries = []
audit_entries = []
stats = {"events": 0}

def logger(event):
    """Log events"""
    log_entries.append(f"LOG: {event}")

def auditor(event):
    """Audit events"""
    audit_entries.append(f"AUDIT: {event}")

def counter(event):
    """Count events"""
    stats["events"] += 1
    print(f"Event #{stats['events']}: {type(event).__name__}")

# Save original subscribers
old_subscribers = zope.event.subscribers[:]

# Register multiple handlers
zope.event.subscribers.extend([logger, auditor, counter])

print(f"Registered {len(zope.event.subscribers)} handlers")

# Send events
events_to_send = [
    "User login",
    "File uploaded", 
    "Database query",
    {"action": "purchase", "amount": 99.99}
]

for event in events_to_send:
    zope.event.notify(event)

print(f"\nResults:")
print(f"Log entries: {len(log_entries)}")
print(f"Audit entries: {len(audit_entries)}")
print(f"Event count: {stats['events']}")

print(f"\nSample log entries:")
for entry in log_entries[:3]:
    print(f"  {entry}")

# Restore original subscribers
zope.event.subscribers[:] = old_subscribers

print("Multiple event handlers completed")

## Example 3: Custom event classes

In [ ]:
import zope.event

# Example 3: Custom event classes
print("Custom event classes:")

class UserEvent:
    """Base class for user-related events"""
    def __init__(self, user_id, timestamp=None):
        self.user_id = user_id
        self.timestamp = timestamp or "2024-01-01T00:00:00"
        
    def __repr__(self):
        return f"{self.__class__.__name__}(user_id={self.user_id})"

class LoginEvent(UserEvent):
    """User login event"""
    def __init__(self, user_id, ip_address, timestamp=None):
        super().__init__(user_id, timestamp)
        self.ip_address = ip_address

class PurchaseEvent(UserEvent):
    """User purchase event"""
    def __init__(self, user_id, item_id, amount, timestamp=None):
        super().__init__(user_id, timestamp)
        self.item_id = item_id
        self.amount = amount

# Event handlers for specific event types
processed_events = {"login": [], "purchase": [], "other": []}

def event_classifier(event):
    """Classify events by type"""
    if isinstance(event, LoginEvent):
        processed_events["login"].append(event)
        print(f"Login: User {event.user_id} from {event.ip_address}")
    elif isinstance(event, PurchaseEvent):
        processed_events["purchase"].append(event)
        print(f"Purchase: User {event.user_id} bought {event.item_id} for ${event.amount}")
    else:
        processed_events["other"].append(event)
        print(f"Other event: {event}")

# Register handler
old_subscribers = zope.event.subscribers[:]
zope.event.subscribers.append(event_classifier)

# Create and notify custom events
events = [
    LoginEvent("user123", "192.168.1.100"),
    PurchaseEvent("user456", "item789", 29.99),
    LoginEvent("user789", "10.0.0.1"),
    "Simple string event"
]

for event in events:
    zope.event.notify(event)

print(f"\nEvent classification results:")
print(f"Login events: {len(processed_events['login'])}")
print(f"Purchase events: {len(processed_events['purchase'])}")
print(f"Other events: {len(processed_events['other'])}")

# Restore subscribers
zope.event.subscribers[:] = old_subscribers

print("Custom event classes completed")

## Example 4: Event filtering and conditional handling

In [ ]:
import zope.event

# Example 4: Event filtering and conditional handling
print("Event filtering and conditional handling:")

# Priority-based event handler
priority_events = {"high": [], "medium": [], "low": []}

def priority_handler(event):
    """Handle events based on priority"""
    if hasattr(event, 'priority'):
        priority = event.priority
        priority_events[priority].append(event)
        print(f"{priority.upper()} priority: {event}")
    else:
        priority_events["low"].append(event)
        print(f"LOW priority (default): {event}")

# Error-sensitive handler
error_count = 0

def error_sensitive_handler(event):
    """Count error events"""
    global error_count
    if hasattr(event, 'is_error') and event.is_error:
        error_count += 1
        print(f"ERROR detected #{error_count}: {event}")

# Conditional handler that only processes certain users
admin_actions = []

def admin_only_handler(event):
    """Only process events from admin users"""
    if hasattr(event, 'user_id') and event.user_id.startswith('admin'):
        admin_actions.append(event)
        print(f"Admin action: {event}")

# Custom event classes with filtering attributes
class PriorityEvent:
    def __init__(self, message, priority="medium"):
        self.message = message
        self.priority = priority
        
    def __repr__(self):
        return f"PriorityEvent({self.message}, {self.priority})"

class ErrorEvent:
    def __init__(self, error_msg):
        self.error_msg = error_msg
        self.is_error = True
        
    def __repr__(self):
        return f"ErrorEvent({self.error_msg})"

class AdminEvent:
    def __init__(self, user_id, action):
        self.user_id = user_id
        self.action = action
        
    def __repr__(self):
        return f"AdminEvent({self.user_id}, {self.action})"

# Register conditional handlers
old_subscribers = zope.event.subscribers[:]
zope.event.subscribers.extend([
    priority_handler, 
    error_sensitive_handler, 
    admin_only_handler
])

# Send mixed events
events = [
    PriorityEvent("System startup", "high"),
    PriorityEvent("User login", "medium"),
    ErrorEvent("Database connection failed"),
    AdminEvent("admin_john", "user_deletion"),
    AdminEvent("user_jane", "profile_update"),
    "Regular string event",
    PriorityEvent("Backup completed", "low"),
    ErrorEvent("File not found")
]

for event in events:
    zope.event.notify(event)

print(f"\nFiltering results:")
print(f"High priority: {len(priority_events['high'])}")
print(f"Medium priority: {len(priority_events['medium'])}")
print(f"Low priority: {len(priority_events['low'])}")
print(f"Total errors: {error_count}")
print(f"Admin actions: {len(admin_actions)}")

# Restore subscribers
zope.event.subscribers[:] = old_subscribers

print("Event filtering and conditional handling completed")

## Example 5: Production event system patterns

In [ ]:
import zope.event
import time
import threading
from collections import defaultdict

# Example 5: Production event system patterns
print("Production event system patterns:")

class EventBus:
    """Production-ready event bus with metrics and error handling"""
    
    def __init__(self):
        self.metrics = {
            'events_published': 0,
            'events_processed': 0,
            'errors': 0,
            'handler_execution_times': defaultdict(list)
        }
        self.original_subscribers = []
        self.handlers = []
    
    def register_handler(self, handler, name=None):
        """Register a named handler with error handling"""
        handler_name = name or handler.__name__
        
        def safe_handler(event):
            start_time = time.time()
            try:
                handler(event)
                self.metrics['events_processed'] += 1
            except Exception as e:
                self.metrics['errors'] += 1
                print(f"Handler {handler_name} error: {e}")
            finally:
                execution_time = time.time() - start_time
                self.metrics['handler_execution_times'][handler_name].append(execution_time)
        
        self.handlers.append((handler_name, safe_handler))
        return safe_handler
    
    def start(self):
        """Start the event bus"""
        self.original_subscribers = zope.event.subscribers[:]
        for name, handler in self.handlers:
            zope.event.subscribers.append(handler)
        print(f"Event bus started with {len(self.handlers)} handlers")
    
    def stop(self):
        """Stop the event bus and restore original state"""
        zope.event.subscribers[:] = self.original_subscribers
        print("Event bus stopped")
    
    def publish(self, event):
        """Publish an event with metrics tracking"""
        self.metrics['events_published'] += 1
        zope.event.notify(event)
    
    def get_metrics(self):
        """Get performance metrics"""
        metrics = dict(self.metrics)
        
        # Calculate average execution times
        avg_times = {}
        for handler_name, times in self.metrics['handler_execution_times'].items():
            if times:
                avg_times[handler_name] = sum(times) / len(times)
        metrics['avg_handler_times'] = avg_times
        
        return metrics

# Create production event bus
event_bus = EventBus()

# Define production handlers
audit_log = []
def audit_handler(event):
    """Audit all events"""
    audit_log.append({
        'event': str(event),
        'timestamp': time.time(),
        'type': type(event).__name__
    })

performance_log = []
def performance_handler(event):
    """Track performance events"""
    if hasattr(event, 'performance_data'):
        performance_log.append(event.performance_data)

def error_handler(event):
    """Handle error events"""
    if hasattr(event, 'is_error') and event.is_error:
        # Simulate error handling (e.g., alerting, logging)
        print(f"ALERT: Error event processed - {event}")

def slow_handler(event):
    """Simulate a slow handler"""
    time.sleep(0.001)  # Simulate processing time
    print(f"Slow handler processed: {type(event).__name__}")

# Register handlers
event_bus.register_handler(audit_handler, "audit")
event_bus.register_handler(performance_handler, "performance")  
event_bus.register_handler(error_handler, "error")
event_bus.register_handler(slow_handler, "slow")

# Start event bus
event_bus.start()

# Define production event types
class PerformanceEvent:
    def __init__(self, operation, duration):
        self.performance_data = {
            'operation': operation,
            'duration': duration,
            'timestamp': time.time()
        }
    
    def __repr__(self):
        return f"PerformanceEvent({self.performance_data['operation']})"

class SystemError:
    def __init__(self, message):
        self.message = message
        self.is_error = True
    
    def __repr__(self):
        return f"SystemError({self.message})"

# Simulate production workload
events = [
    PerformanceEvent("database_query", 0.05),
    "User action: login",
    SystemError("Connection timeout"),
    PerformanceEvent("api_call", 0.12),
    "User action: logout",
    SystemError("Invalid credentials"),
    PerformanceEvent("file_upload", 0.8)
]

print(f"\nProcessing {len(events)} events...")
for event in events:
    event_bus.publish(event)

# Show results
metrics = event_bus.get_metrics()
print(f"\nEvent Bus Metrics:")
print(f"  Events published: {metrics['events_published']}")
print(f"  Events processed: {metrics['events_processed']}")  
print(f"  Errors: {metrics['errors']}")

print(f"\nHandler Performance:")
for handler, avg_time in metrics['avg_handler_times'].items():
    print(f"  {handler}: {avg_time*1000:.2f}ms avg")

print(f"\nAudit Log: {len(audit_log)} entries")
print(f"Performance Log: {len(performance_log)} entries")

# Stop event bus
event_bus.stop()

print(f"\nProduction patterns demonstrated:")
print("- Error handling and recovery")
print("- Performance monitoring and metrics")
print("- Handler registration and lifecycle management")
print("- Event bus architecture with cleanup")
print("- Audit logging and compliance")

print("Production event system patterns completed")